# Unified Model Benchmark — all models over the `mtq-refactor` project

Runs all five available risk models over the `calibration` and `validation`
datasets, producing probability-map rasters for each, then a side-by-side
visual comparison.

Models split into two families with different `fit`/`apply` interfaces:

| Family | Models | `fit()` | `apply()` |
|--------|--------|---------|-----------|
| ML | `GLMModel`, `RFModel`, `ICARModel` | uses attached `dataset` + `sampling` | `apply(output_file, dataset, mask, mask_value)` |
| Benchmark | `MWModel`, `JNRBenchmarkModel` | `fit(dataset=, …, folder=)` | `apply(dataset=, output_folder=, time_interval=)` → dict{window→Path} |

Each model is fit on **calibration** and applied to **both** periods.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys

sys.path.append("..")

from spatialrisk import (
    Project,
    Dataset,
    GLMModel,
    RFModel,
    ICARModel,
    MWModel,
    JNRBenchmarkModel,
    rmj,
)
from spatialrisk.sampling import Sampling, SamplingStrategy

In [ ]:
project_name = "mtq-refactor"
project = Project.load(project_name=project_name)
project.list_datasets()

In [ ]:
calibration = project.get_dataset("calibration")
validation = project.get_dataset("validation")
calibration.validate()

### Shared parameters

`periods` maps each dataset to its time interval in years
(calibration = 2020 − 2015 = 5, validation = 2024 − 2020 = 4).
`forest_mask` is the forest raster used to mask ML-model predictions.
`results` accumulates one representative probability raster per
`(model, period)` for the comparison grid at the end.

In [ ]:
random_seed = 1

sampling = Sampling(
    strategy=SamplingStrategy.legacy,
    n_samples=10000,
    seed=33,            # for reproducibility
    adapt=True,
    pixel_area_ha=0.09,
)

# period name -> (dataset, time_interval_years)
periods = {
    "calibration": (calibration, 5),
    "validation": (validation, 4),
}

# Forest raster used as the mask for ML-model apply()
forest_mask = calibration.get_file_paths()["forest_gfc"]

# (model_name, period_name) -> Path to a single representative probability raster
results = {}

## 2. ML models

### 2.1 GLM — logistic regression

Constructed with the calibration dataset + sampling attached; `fit()` trains on
calibration, then `apply()` predicts a probability raster for each period,
masked to forest (`mask_value=0`).

In [ ]:
glm = GLMModel(
    name="glm_v1",
    dataset=calibration,
    sampling=sampling,
    random_seed=random_seed,
)
glm.register(project)
glm.fit()

In [ ]:
for period, (ds, _ti) in periods.items():
    out = project.folders.glm_model / glm.name / f"{period}.tif"
    glm.apply(out, ds, forest_mask, 0)
    results[("GLM", period)] = out
    print(f"GLM {period:12s} → {out}")

### 2.2 Random Forest

In [ ]:
rf = RFModel(
    name="rf_v1",
    dataset=calibration,
    sampling=sampling,
    random_seed=random_seed,
)
rf.register(project)
rf.fit()

In [ ]:
for period, (ds, _ti) in periods.items():
    out = project.folders.rf_model / rf.name / f"{period}.tif"
    rf.apply(out, ds, forest_mask, 0)
    results[("RF", period)] = out
    print(f"RF  {period:12s} → {out}")

### 2.3 iCAR — spatial logistic regression (MCMC)

In [ ]:
icar = ICARModel(
    name="icar_v1",
    dataset=calibration,
    sampling=sampling,
    random_seed=random_seed,
)
icar.register(project)
icar.fit()

In [ ]:
for period, (ds, _ti) in periods.items():
    out = project.folders.icar_model / icar.name / f"{period}.tif"
    icar.apply(out, ds, forest_mask, 0)
    results[("ICAR", period)] = out
    print(f"ICAR {period:11s} → {out}")

## 3. Benchmark models

### 3.1 Moving Window (MW)

Fit on calibration, then applied to each period. `apply()` returns a dict
mapping each window size to a probability raster; we keep window **11** as the
representative map for the comparison grid.

In [ ]:
WIN_REPR = 11  # representative window size for the comparison grid

mw = MWModel(
    name="calibration_mw",
    forest_edge_var="forest_gfc_edge",
    forest_var="forest_gfc",
    win_size_list=[5, 11, 21],
    defor_threshold=99.5,
    max_dist=50000,
    blk_rows=256,
)
mw.fit(dataset=calibration, time_interval=5, folder=project.folders.rmj_mw)
mw.register(project)
print(f"dist_thresh   : {mw.dist_thresh:.1f} m")
print(f"ldefrate_files: {list(mw.ldefrate_files.keys())} window sizes")

In [ ]:
for period, (ds, ti) in periods.items():
    outputs = mw.apply(
        dataset=ds,
        time_interval=ti,
        output_folder=project.folders.rmj_mw,
    )
    results[("MW", period)] = outputs[str(WIN_REPR)]  # MW keys are strings
    print(f"MW  {period:12s} (win {WIN_REPR}) → {outputs[str(WIN_REPR)].name}")

### 3.2 JNR benchmark

Unlike MW, JNR's `apply()` writes a **single** vulnerability raster per period
and returns its `Path`. The validation period reuses the calibration period's
deforestation-rate table (`deforate_model=`), so calibration must run first.

In [ ]:
jnr = JNRBenchmarkModel(
    name="calibration_jnr",
    forest_edge_var="forest_gfc_edge",
    forest_var="forest_gfc",
    subj_var="subj",
    defor_threshold=99.5,
    max_dist=50000,
    blk_rows=128,
)
jnr.fit(dataset=calibration, defor_threshold=99.5, folder=project.folders.rmj_bm)
jnr.register(project)
print(f"dist_thresh : {jnr.dist_thresh:.1f} m")
print(f"dist_bins   : {len(jnr.dist_bins)} edges → {len(jnr.dist_bins) - 1} classes")

In [ ]:
# Calibration must run before validation: validation reuses the calibration
# defrate table for quantity-adjustment.
for period, (ds, ti) in periods.items():
    out = project.folders.rmj_bm / period / f"prob_bm_{period}.tif"
    deforate_model = (
        None if period == "calibration" else jnr.defrate_files.get("calibration")
    )
    jnr.apply(
        output_file=out,
        dataset=ds,
        time_interval=ti,
        deforate_model=deforate_model,
    )
    results[("JNR", period)] = out
    print(f"JNR {period:12s} → {out.name}")

## 4. Visual comparison

A 5-column (one per model) × 2-row (calibration, validation) grid of the
probability rasters, on a shared colormap and color scale. The helper reads
each raster, masks its nodata value (ML rasters use nodata=0, benchmark rasters
use 65535), and normalizes to [0, 1] for display.

In [ ]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt


def load_prob(path):
    """Read a probability raster, mask nodata, normalize to [0, 1]."""
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        nodata = src.nodata
    # Mask known nodata conventions: 0 (ML) and 65535 (benchmark)
    mask = np.isnan(arr)
    if nodata is not None:
        mask |= arr == nodata
    mask |= arr == 0
    mask |= arr == 65535
    arr = np.ma.masked_array(arr, mask=mask)
    vmax = arr.max()
    if vmax and vmax > 0:
        arr = arr / vmax
    return arr

In [ ]:
model_order = ["GLM", "RF", "ICAR", "MW", "JNR"]
period_order = ["calibration", "validation"]

fig, axes = plt.subplots(
    len(period_order),
    len(model_order),
    figsize=(4 * len(model_order), 4 * len(period_order)),
)
cmap = plt.get_cmap("viridis").copy()
cmap.set_bad("lightgrey")

for r, period in enumerate(period_order):
    for c, model_name in enumerate(model_order):
        ax = axes[r, c]
        ax.set_xticks([])
        ax.set_yticks([])
        path = results.get((model_name, period))
        if path is None:
            ax.text(0.5, 0.5, "no output", ha="center", va="center")
            continue
        im = ax.imshow(load_prob(path), cmap=cmap, vmin=0, vmax=1)
        if r == 0:
            ax.set_title(model_name, fontsize=13)
        if c == 0:
            ax.set_ylabel(period, fontsize=12)

fig.colorbar(im, ax=axes, shrink=0.6, label="relative risk (normalized)")
fig.suptitle("Model probability maps — mtq-refactor", fontsize=15, y=0.98)
plt.show()